# Домашнее задание 6: Segment Anything

**Цель:** разобраться, как SAM ведёт себя на реальных данных: от чего зависит качество масок, какие подсказки лучше работают
и можно ли использовать SAM как **разметчик** для обучения своей модели.

Первая часть ноутбука (пункты 0–5) — **разобранный семинар**, его нужно просто запустить: функции и переменные оттуда понадобятся в заданиях.
Задания — в конце ноутбука (раздел «Задания»).

**Где запускать:** всё рассчитано на **Google Colab с GPU T4** (Runtime → Change runtime type → T4 GPU). Ориентиры по времени на T4 приведены в каждом задании.

## Система оценки (макс. 10 баллов)

| задание | баллы |
|---|---|
| 1. Параметры автоматической разметки | 2 |
| 2. Стратегии подсказок | 2 |
| 3. SAM как разметчик: псевдо-маски для U-Net | 4 |
| 4. SAM vs SAM 2 | 2 |

**+1 балл** может добавить ревьюер за интересный анализ, визуализацию или нестандартное решение.

Требования ко всем заданиям:
- код должен запускаться сверху вниз без ошибок;
- в каждом задании — **таблица/график с результатами** и **текстовые выводы** (2–5 предложений): что получилось и почему;
- баллы снижаются за отсутствие выводов, даже если код работает.

In [ ]:
# Для Google Colab: скачиваем репозиторий курса (картинки из assets/) и ставим зависимости.
# Локально эту ячейку можно пропустить.
%cd /content
!rm -rf cv-course-at-mipt-fall-2026
!git clone --depth 1 https://github.com/T-CV/cv-course-at-mipt-fall-2026.git
%cd cv-course-at-mipt-fall-2026/lesson_6
!pip install -q segment-anything segmentation-models-pytorch

## Часть 1. Разобранный семинар

**План семинара:**

1. Загружаем SAM и смотрим, почему он интерактивный: один embedding — много подсказок
2. Подсказки: точка, неоднозначность и 3 маски
3. Уточнение маски: отрицательные точки, бокс, mask prompt
4. Насколько SAM хорош «из коробки»: сравнение с GT на Oxford-IIIT Pet
5. Автоматическая разметка всей картинки + постобработка масок морфологией

Теория — в `6-3 SAM.ipynb`.

## 0. Подготовка

```bash
pip install segment-anything opencv-python torchvision matplotlib
```

Чекпойнт SAM ViT-B (~375 МБ) скачается в `assets/` при первом запуске.

**Устройство:** на CUDA всё быстро. На Apple Silicon (MPS) `SamPredictor` работает, но `SamAutomaticMaskGenerator` падает
(внутри используется float64, MPS его не поддерживает), поэтому без CUDA используем CPU: embedding картинки ~3–5 с, подсказка — доли секунды.

In [ ]:
import time
import urllib.request
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.axes import Axes
from PIL import Image
from scipy import ndimage as ndi

import torch
import torchvision
from segment_anything import SamAutomaticMaskGenerator, SamPredictor, sam_model_registry

%matplotlib inline

device = "cuda" if torch.cuda.is_available() else "cpu"

CHECKPOINT = Path("assets/sam_vit_b_01ec64.pth")
if not CHECKPOINT.exists():
    print("скачиваем чекпойнт SAM ViT-B ...")
    urllib.request.urlretrieve("https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth", CHECKPOINT)

sam = sam_model_registry["vit_b"](checkpoint=str(CHECKPOINT)).to(device).eval()
predictor = SamPredictor(sam)
print("device:", device)

In [ ]:
def show_mask(mask: np.ndarray, ax: Axes, color: np.ndarray | None = None) -> None:
    color = np.array([30 / 255, 144 / 255, 1.0, 0.6]) if color is None else color
    ax.imshow(mask[..., None] * color.reshape(1, 1, 4))


def show_points(coords: np.ndarray, labels: np.ndarray, ax: Axes) -> None:
    for lbl, color in [(1, "lime"), (0, "red")]:
        pts = coords[labels == lbl]
        ax.scatter(pts[:, 0], pts[:, 1], color=color, marker="*", s=300, edgecolor="white", linewidth=1.25)


def show_box(box: np.ndarray, ax: Axes) -> None:
    x0, y0, x1, y1 = box
    ax.add_patch(plt.Rectangle((x0, y0), x1 - x0, y1 - y0, edgecolor="lime", facecolor="none", lw=2))


def iou(a: np.ndarray, b: np.ndarray) -> float:
    return (a & b).sum() / max((a | b).sum(), 1)


img = np.array(Image.open("assets/food.jpg").convert("RGB"))
print("image:", img.shape)

## 1. Один embedding — много подсказок

`set_image` прогоняет тяжёлый image encoder (ресайз длинной стороны до 1024 → ViT → embedding $64 \times 64 \times 256$).
`predict` запускает только prompt encoder + mask decoder. Сравним время.

In [ ]:
start = time.time()
predictor.set_image(img)
t_encoder = time.time() - start

point = np.array([[440, 520]])  # (x, y) — кусочек дыни в миске
start = time.time()
for _ in range(10):
    masks, scores, low_res_logits = predictor.predict(point_coords=point, point_labels=np.array([1]), multimask_output=True)
t_decoder = (time.time() - start) / 10

print(f"image encoder: {t_encoder:.2f} s | prompt encoder + mask decoder: {t_decoder * 1000:.0f} ms")
print("embedding:", tuple(predictor.get_image_embedding().shape))
print("masks:", masks.shape, "| scores:", scores.round(3), "| low-res logits:", low_res_logits.shape)

## 2. Неоднозначность: одна точка — три маски

Точка на ободке миски: это миска? тарелка вместе с миской? SAM выдаёт 3 гипотезы и для каждой — **предсказанный IoU** (score).

In [ ]:
def show_multimask(img: np.ndarray, point: np.ndarray) -> None:
    masks, scores, _ = predictor.predict(point_coords=point, point_labels=np.array([1]), multimask_output=True)
    fig, axs = plt.subplots(1, 3, figsize=(15, 6))
    for i, ax in enumerate(axs):
        ax.imshow(img)
        show_mask(masks[i], ax)
        show_points(point, np.array([1]), ax)
        ax.set_title(f"mask {i}: score={scores[i]:.3f}, area={masks[i].mean():.1%}")
        ax.axis("off")
    plt.show()


show_multimask(img, np.array([[370, 640]]))  # ободок миски
show_multimask(img, np.array([[440, 520]]))  # кусочек дыни

**Вопросы:**
- Всегда ли маска с максимальным score — «та, что нужна»? Score — это оценка качества маски, а не «правильности намерения».
- Попробуйте точки на хлебе, на клубнике, на краю тарелки. Где 3 маски сильно различаются, а где почти одинаковы?

## 3. Уточнение маски

Инструменты, которые SAM видел при обучении (в 11 раундах имитации разметки):
- **отрицательные точки** — «это не включать»;
- **бокс** — сильная подсказка, почти снимает неоднозначность;
- **mask prompt** — подаём логиты предыдущего шага (`low_res_logits`, 256×256), и модель уточняет собственную маску.

При нескольких подсказках неоднозначности меньше, поэтому используем `multimask_output=False`.

In [ ]:
fig, axs = plt.subplots(1, 4, figsize=(22, 7))

# a) одна точка на кусочке дыни: лучшая маска по score — один кусочек
p1, l1 = np.array([[440, 520]]), np.array([1])
masks, scores, logits = predictor.predict(point_coords=p1, point_labels=l1, multimask_output=True)
best = scores.argmax()
panels = [(masks[best], p1, l1, None, f"1 точка, score={scores[best]:.3f}")]

# b) хотим больше фруктов: + 2 положительные точки, + mask prompt с предыдущего шага
p2, l2 = np.array([[440, 520], [640, 560], [500, 720]]), np.array([1, 1, 1])
m2, s2, _ = predictor.predict(point_coords=p2, point_labels=l2, mask_input=logits[best][None], multimask_output=False)
panels.append((m2[0], p2, l2, None, f"3 точки + mask prompt, score={s2[0]:.3f}"))

# c) бокс вокруг миски — выделяется миска целиком, вместе с фруктами
box = np.array([345, 400, 728, 815])
m3, s3, _ = predictor.predict(box=box, multimask_output=False)
panels.append((m3[0], None, None, box, f"бокс, score={s3[0]:.3f}"))

# d) бокс + отрицательная точка на ободке: "в этом боксе, но без миски" -> только фрукты
p4, l4 = np.array([[370, 640]]), np.array([0])
m4, s4, _ = predictor.predict(point_coords=p4, point_labels=l4, box=box, multimask_output=False)
panels.append((m4[0], p4, l4, box, f"бокс + отриц. точка, score={s4[0]:.3f}"))

for ax, (m, p, l, b, title) in zip(axs, panels):
    ax.imshow(img)
    show_mask(m, ax)
    if p is not None:
        show_points(p, l, ax)
    if b is not None:
        show_box(b, ax)
    ax.set_title(title)
    ax.axis("off")

## 4. SAM против GT

Насколько хороши маски «из коробки»? Возьмём **Oxford-IIIT Pet** (тот же датасет, что в примере обучения U-Net в лекции, ~800 МБ в `data/`)
и сравним разные подсказки с GT-маской (животное + граница trimap):

1. **1 точка** в «самом центре» объекта (максимум distance transform GT) → маска с максимальным score;
2. **oracle:** из 3 масок берём лучшую по IoU с GT — верхняя граница, если бы человек выбрал гипотезу сам;
3. **бокс** GT-маски;
4. **бокс + точка**.

> GT-маска включает полосу «границы» trimap, а шерсть и уши размечены не идеально — абсолютные цифры не надо воспринимать как точные.

In [ ]:
pets = torchvision.datasets.OxfordIIITPet("data", split="test", target_types="segmentation", download=True)
rng = np.random.default_rng(0)
indices = rng.choice(len(pets), size=30, replace=False)

results = {"1 точка (best score)": [], "1 точка (oracle)": [], "бокс": [], "бокс + точка": []}
examples = []

for idx in indices:
    image, trimap = pets[int(idx)]
    image, gt = np.array(image.convert("RGB")), np.array(trimap) != 2
    predictor.set_image(image)

    dist = ndi.distance_transform_edt(gt)
    cy, cx = np.unravel_index(dist.argmax(), dist.shape)
    point, label_1 = np.array([[cx, cy]]), np.array([1])
    ys, xs = np.nonzero(gt)
    box = np.array([xs.min(), ys.min(), xs.max(), ys.max()])

    masks, scores, _ = predictor.predict(point_coords=point, point_labels=label_1, multimask_output=True)
    ious = [iou(m, gt) for m in masks]
    results["1 точка (best score)"].append(ious[scores.argmax()])
    results["1 точка (oracle)"].append(max(ious))

    m_box, _, _ = predictor.predict(box=box, multimask_output=False)
    results["бокс"].append(iou(m_box[0], gt))

    m_bp, _, _ = predictor.predict(point_coords=point, point_labels=label_1, box=box, multimask_output=False)
    results["бокс + точка"].append(iou(m_bp[0], gt))

    examples.append((image, gt, masks[scores.argmax()], point, ious[scores.argmax()]))

for name, values in results.items():
    print(f"{name:>22}: mean IoU = {np.mean(values):.3f}, median = {np.median(values):.3f}, min = {np.min(values):.3f}")

In [ ]:
# 4 худших примера для подсказки "1 точка"
worst = sorted(examples, key=lambda e: e[-1])[:4]
fig, axs = plt.subplots(2, 4, figsize=(20, 9))
for i, (image, gt, pred, point, score) in enumerate(worst):
    axs[0, i].imshow(image)
    show_mask(gt, axs[0, i], color=np.array([0.1, 0.9, 0.1, 0.5]))
    axs[0, i].set_title("GT")
    axs[1, i].imshow(image)
    show_mask(pred, axs[1, i])
    show_points(point, np.array([1]), axs[1, i])
    axs[1, i].set_title(f"SAM, 1 точка: IoU={score:.3f}")
for ax in axs.flat:
    ax.axis("off")

**Обсуждение:**
- Сравните с U-Net, обученным в лекции на этом же датасете. Что лучше — специализированная модель за несколько минут обучения или zero-shot SAM с подсказкой? А если подсказок нет?
- Посмотрите на худшие примеры: какую «гипотезу» выбрал SAM вместо животного целиком? Помог бы oracle-выбор из 3 масок или бокс?
- Отсюда типичный приём: **SAM + бокс от детектора** = instance segmentation без обучения сегментационной модели, или **SAM как разметчик** (псевдо-маски по боксам).

## 5. Автоматическая разметка всей картинки

`SamAutomaticMaskGenerator` — это стадия fully automatic из data engine:
1. сетка `points_per_side × points_per_side` точек-подсказок;
2. для каждой — 3 маски; фильтр по предсказанному IoU (`pred_iou_thresh`) и **стабильности** (`stability_score_thresh`:
   IoU между масками, бинаризованными при порогах логитов $\pm$ offset — устойчивые маски почти не меняются);
3. NMS по боксам, опционально — удаление мелких дырок/островков (`min_mask_region_area`).

По умолчанию `points_per_side=32` (1024 подсказки). На CPU это долго — для семинара берём 16.

In [ ]:
mask_generator = SamAutomaticMaskGenerator(sam, points_per_side=16, pred_iou_thresh=0.88, stability_score_thresh=0.95)

start = time.time()
auto_masks = mask_generator.generate(img)
print(f"масок: {len(auto_masks)} за {time.time() - start:.1f} s")
print("поля одной маски:", sorted(auto_masks[0].keys()))


def show_all_masks(img: np.ndarray, masks: list[dict], ax: Axes, title: str) -> None:
    ax.imshow(img)
    rng = np.random.default_rng(0)
    for m in sorted(masks, key=lambda m: m["area"], reverse=True):  # большие снизу, мелкие поверх
        show_mask(m["segmentation"], ax, color=np.concatenate([rng.random(3), [0.55]]))
    ax.set_title(f"{title}: {len(masks)}")
    ax.axis("off")

### Постобработка масок

Маски SAM часто содержат **мелкие отверстия** и **оторванные островки**. Простые инструменты:
- фильтр по площади (`area`) и по `predicted_iou` / `stability_score`;
- **морфология:** closing (dilate → erode) закрывает дырки, opening (erode → dilate) убирает мелкий шум.
  Размер ядра — гиперпараметр: слишком большое ядро «съедает» тонкие детали.

In [ ]:
def filter_masks(masks: list[dict], min_area: int = 1500) -> list[dict]:
    return [m for m in masks if m["area"] >= min_area]


def clean_mask(mask: np.ndarray, kernel_size: int = 7) -> np.ndarray:
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (kernel_size, kernel_size))
    m = mask.astype(np.uint8)
    m = cv2.morphologyEx(m, cv2.MORPH_CLOSE, kernel)  # закрываем дырки
    m = cv2.morphologyEx(m, cv2.MORPH_OPEN, kernel)   # убираем островки
    return m.astype(bool)


filtered = filter_masks(auto_masks)
cleaned = [{**m, "segmentation": clean_mask(m["segmentation"])} for m in filtered]  # исходные маски не трогаем

fig, axs = plt.subplots(1, 3, figsize=(21, 9))
show_all_masks(img, auto_masks, axs[0], "Все маски SAM")
show_all_masks(img, filtered, axs[1], "area ≥ 1500")
show_all_masks(img, cleaned, axs[2], "+ closing / opening")

In [ ]:
# Что делает морфология с одной маской: число связных компонент и дырок до/после
def n_components(mask: np.ndarray) -> int:
    return ndi.label(mask)[1]


def n_holes(mask: np.ndarray) -> int:
    return ndi.label(~mask)[1] - 1  # компоненты фона минус "внешний" фон


worst = max(filtered, key=lambda m: n_components(m["segmentation"]) + n_holes(m["segmentation"]))
before, after = worst["segmentation"], clean_mask(worst["segmentation"])
print(f"до:    компонент {n_components(before)}, дырок {n_holes(before)}")
print(f"после: компонент {n_components(after)}, дырок {n_holes(after)}")

x0, y0, w, h = map(int, worst["bbox"])
fig, axs = plt.subplots(1, 2, figsize=(12, 6))
for ax, m, title in zip(axs, [before, after], ["до", "после closing / opening"]):
    ax.imshow(m[y0:y0 + h, x0:x0 + w], cmap="gray")
    ax.set_title(title)
    ax.axis("off")

# Задания

## Задание 1. Параметры автоматической разметки (2 балла)

`SamAutomaticMaskGenerator` из пункта 5 сильно зависит от параметров. Исследуйте, как на одной картинке (`assets/food.jpg` + одна картинка Pet на ваш выбор) меняются результаты при разных:
- `points_per_side` ∈ {8, 16, 32};
- `pred_iou_thresh` ∈ {0.80, 0.88, 0.95};
- `stability_score_thresh` ∈ {0.85, 0.95}.

Для каждой конфигурации посчитайте: **число масок**, **время** генерации, **покрытие** — долю пикселей, попавших хотя бы в одну маску.
Сведите в таблицу, визуализируйте 3–4 характерные конфигурации.

**Оценивание:**
- 1 балл — таблица по всем конфигурациям;
- 1 балл — визуализация + выводы: какой параметр сильнее всего влияет на число / качество масок, какой компромисс «скорость — полнота» вы бы выбрали.

*Время на T4: одна генерация с `points_per_side=32` — секунды; вся сетка — несколько минут.*

In [ ]:
# YOUR CODE HERE

## Задание 2. Стратегии подсказок (2 балла)

В пункте 4 мы сравнили «1 точку», «бокс» и «бокс + точку». Расширьте эксперимент на **100 картинках** `split="test"` Pet:

1. **k положительных точек** внутри GT-маски, $k \in \{1, 3, 5\}$ (как выбирать точки — решите сами: случайно, по distance transform, …);
2. **отрицательные точки** из фона: 1 точка в центре + 1–4 отрицательные (например, в углах GT-бокса, если они вне маски);
3. **бокс с шумом**: GT-бокс, каждая координата которого сдвинута на случайные ±5% / ±10% / ±20% от размера бокса — насколько SAM устойчив к неточному детектору?

Для каждой стратегии — mean и median IoU с GT. Для неоднозначных подсказок (одна точка) сравните выбор маски по `score` и oracle-выбор по IoU с GT.

**Оценивание:**
- 1 балл — пункты 1 и 2 с таблицей;
- 1 балл — пункт 3 (график IoU от величины шума) и выводы: какая подсказка даёт лучший результат «за клик».

*Время на T4: `set_image` на картинку — доли секунды, 100 картинок × все подсказки — несколько минут (embedding считайте один раз на картинку!).*

In [ ]:
# YOUR CODE HERE

## Задание 3. SAM как разметчик: псевдо-маски для U-Net (4 балла)

Типичный практический сценарий: боксы разметить дёшево, маски — дорого. Проверим, можно ли получить маски с помощью SAM.

1. Возьмите **1000 картинок** `split="trainval"` Pet (фиксированный seed) и **400** других картинок для валидации.
2. Для train-картинок сгенерируйте **псевдо-маски**: SAM + GT-бокс (+ при желании точка, см. задание 2).
   Посчитайте качество псевдо-масок: IoU с настоящими масками.
3. Обучите **две одинаковые модели** U-Net (как в лекции `6-1`, раздел 4.7: `smp.Unet("resnet34")`, 256×256, BCE + Dice):
   - модель A — на **настоящих** масках;
   - модель B — на **псевдо-масках** SAM.

   Одинаковые гиперпараметры, seed и число эпох (3–5 достаточно).
4. Сравните обе модели на валидации **по настоящим маскам**. Покажите примеры, где модель B заметно хуже.

**Оценивание:**
- 1 балл — генерация псевдо-масок и оценка их качества;
- 1 балл — корректный пайплайн обучения двух моделей (одинаковые условия, валидация по GT);
- 1 балл — сравнение и визуализация ошибок;
- 1 балл — выводы: насколько «просела» модель B относительно качества самих псевдо-масок, где SAM систематически ошибается (подсказка: посмотрите на trimap-границу и тонкие части — уши, хвосты, лапы).

*Время на T4: псевдо-маски для 1000 картинок — несколько минут; одна эпоха U-Net на 1000 картинках 256×256 — около минуты.*

In [ ]:
# YOUR CODE HERE

## Задание 4. SAM vs SAM 2 (2 балла)

Сравните SAM ViT-B с **SAM 2** на картинках. SAM 2 удобно брать из 🤗 `transformers` (`Sam2Model`, `Sam2Processor`,
например чекпойнт `facebook/sam2.1-hiera-tiny` или `-small`) — разберитесь с API по документации.

На тех же 100 картинках, что в задании 2, с подсказками **бокс** и **1 точка** сравните:
- mean / median IoU с GT;
- время на картинку: отдельно encoder (embedding) и decoder (подсказка).

**Оценивание:**
- 1 балл — корректный запуск SAM 2 и таблица метрик;
- 1 балл — замеры времени (не забудьте `torch.cuda.synchronize()` и прогрев) + выводы: что вы бы выбрали для интерактивной разметки и почему.

*Время на T4: несколько минут; модели tiny/small занимают сотни МБ.*

In [ ]:
# YOUR CODE HERE